In [ ]:
#| default_exp skill

# nbskill Pyskill

> Context search, structural review, clean diffs, and nbdev authoring for Python coding agents.

#| export
Use `nbskill.skill` for nbdev notebooks and their generated `.py` modules (`from nbskill.skill import *`). Works standalone; alongside `aai-coding`, use `nbskill.skill` for notebook maps, symbol/chapter bundles, structural review, clean diffs, and execution freshness while keeping `exhash`, `rgapi`, and `nbdev-test` for edits, fast grep, and full test runs.

## Choose a function

| Question / task | Call |
|---|---|
| Map notebooks, H2 chapters, symbols, hidden % | `nb_map(path=".")` |
| Route a `.py` file back to its source notebook | `generated_owner(py_path)` |
| Read one H2 chapter with IDs, hashes, freshness | `chapter_context(path, chapter)` |
| Inspect a symbol's prose, code, examples, callers | `symbol_context(target, scope=".")` |
| Regex search cells standalone (smart-case) | `search_nb(pattern, path=".")` |
| Catch structural, state, narrative, and style issues | `review_nb(path=".")` / `check_nb(path=".")` |
| Cell-level diff (code + Markdown, no JSON/output noise) | `diff_nb(path, ref_a="HEAD")` |
| Inspect clean/dirty/stale execution state | `notebook_state(path)` |
| Atomic cell edits standalone (strict hash/match checks) | `edit_nb(path, ops, dry_run=False)` |
| Run top through a chapter or cell ID standalone | `exec_nb(path, chapter=..., save_outputs=False)` |

## Writing readable nbdev notebooks

- Structure: one topic per notebook; one concept per H2 (`##`) chapter; H1 title + `>` subtitle at top; each heading in its own Markdown cell so Jupyter collapses sections cleanly.
- Prose: lean, direct sentences explaining contracts, tradeoffs, and edge cases; never narrate code line by line.
- Lessons: keep definition, explanation, visible example, and focused assertion inside the same H2 chapter; group related helpers under one shared chapter lesson instead of writing filler tests per trivial helper.
- Idioms: develop class methods beside their explanation with `@patch`; end visible examples with a bare value (not `print(x)`); verify errors with `expect_fail` or `test_fail` (not `try`/`except`).
- State: dedicated cells for imports (exported vs lesson); never depend on `#| eval: false` bindings or silently reassign shared names read earlier.
- Visibility: keep implementation visible and explained; `#| hide` is for genuine distracting setup only, never to conceal logic or silence style checks.
- Edit loop: `generated_owner` -> `nb_map` / `symbol_context` -> edit notebook (never `# AUTOGENERATED!` `.py`) -> `exec_nb` / `nbdev-test` -> `review_nb` + `diff_nb` -> `nbdev-export`.

In [ ]:
#| export
from nbskill.context import chapter_context, nb_map, search_nb, symbol_context
from nbskill.core import generated_owner
from nbskill.review import check_nb, review_nb
from nbskill.state import diff_nb, notebook_state
from nbskill.work import edit_nb, exec_nb

Declare `_all_` so nbdev exports the re-imported Pyskill API cleanly for `from nbskill.skill import *`.

In [ ]:
#| export
_all_ = [nb_map, generated_owner, chapter_context, symbol_context, search_nb,
    review_nb, check_nb, diff_nb, notebook_state, edit_nb, exec_nb]

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq
from pyskills import list_pyskills

## Orient and understand

Start with `generated_owner` and `nb_map`, then pull one symbol's complete story with `symbol_context`.

In [ ]:
#| exportd
nbs_dir = Path("nbs") if Path("nbs").is_dir() else Path(".")
owner = generated_owner(Path("nbskill/core.py")) if Path("nbskill/core.py").is_file() else None
proj = nb_map(nbs_dir)
assert proj.notebooks
if owner is not None: test_eq(owner.name, "00_core.ipynb")
proj

`symbol_context` retrieves the owning chapter, explanation, definition, nearby example, and callers in one call.

In [ ]:
#| exportd
ctx = symbol_context("generated_owner", scope=nbs_dir)
test_eq(ctx.chapter, "Project notebooks and source ownership")
assert ctx.definitions and ctx.examples
ctx

## Edit, execute, diff, and review

A disposable notebook proves the full standalone loop: edit atomically, run to record freshness, inspect the clean diff, and pass `review_nb`.

In [ ]:
#| exportd
with TemporaryDirectory() as tmp:
    p, c_fn, c_ex = Path(tmp) / "lesson.ipynb", mk_cell("#| export\ndef double(x): return x * 2"), mk_cell("ans = double(20)\nassert ans == 42\nans")
    write_nb(new_nb([mk_cell("# Lesson", "markdown"), mk_cell("> Small example.", "markdown"),
        mk_cell("## Double", "markdown"), mk_cell("`double` multiplies a number by two.", "markdown"), c_fn, c_ex]), p)
    res_edit = edit_nb(p, [dict(op="replace_text", cell_id=c_ex.id, old="double(20)", new="double(21)")])
    res_run, res_rev = exec_nb(p, save_outputs=True), review_nb(p)
    test_eq((res_edit.changed, res_run.ok, res_rev.ok, notebook_state(p).counts["clean"]), (True, True, True, 2))
    assert "nbskill.skill" in list_pyskills()
res_rev